In [1]:
1

1

In [2]:
import os
from dotenv import load_dotenv

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [3]:
# load_dotenv()
# api_key = os.getenv('NVIDIA_API_KEY')

In [4]:
llm = ChatOpenAI(
    model='gpt-5-mini',
    temperature=0.2,
    reasoning_effort='medium'
)

In [5]:
rt = llm.invoke('안녕')

In [6]:
rt.content

'안녕! 반가워. 뭐 도와줄까?'

In [7]:
for chunk in llm.stream("오늘 저녁 추천해줘"):
    print(chunk.content , end="", flush=True)


오늘 저녁 어떠한 분위기 원해? 빠르게? 건강하게? 매콤하게? 일단 몇 가지 골라봤어 — 원하면 바로 레시피나 장보기 목록도 줄게.

1) 비빔밥 — 균형 잡힌 한 끼, 남은 채소 활용 가능. 시간 20–30분, 난이도 쉬움.  
2) 김치찌개 — 따끈하고 든든한 매운 국물요리. 시간 25–35분, 난이도 쉬움.  
3) 제육볶음 + 밥 — 맵고 푸짐한 메뉴, 술안주로도 좋아. 시간 20–30분, 난이도 보통.  
4) 연어 구이와 샐러드 — 건강하고 가벼운 저녁, 준비 빠름. 시간 15–20분, 난이도 쉬움.  
5) 알리오 올리오(파스타) — 재료 적고 빠르게 만드는 이탈리안. 시간 10–15분, 난이도 쉬움.  
6) 배달 치킨 또는 피자 — 요리하기 귀찮을 때 안성맞춤. 시간: 주문 후 배달 시간에 따라 다름.

어떤 걸로 할래? 하나 골라주면 재료와 간단한 조리 순서 알려줄게. 혹시 채식/알레르기/피망 싫음 같은 제한 있으면 알려줘.

In [8]:
import requests
url = "https://www.melon.com/song/detail.htm?songId=32496320"
head = {
    'user-agent':
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0"
}


from bs4 import BeautifulSoup
bs = BeautifulSoup(requests.get(url, headers=head).text)


raw_lyrics = bs.find('div', class_='lyric')

In [9]:
lyrics = BeautifulSoup(str(raw_lyrics).replace('<br/>', '\n')).text.replace('\t', '')

In [10]:
prompt = PromptTemplate.from_template(
    """
    아래 제공된 노래 가사를 바탕으로 해당 가사 내용을 비둘기에 사랑이 표현되게
    변경해줘
    {lyrics}
    """
)


In [11]:
chain = prompt | llm | StrOutputParser()
rt2 = chain.invoke({'lyrics' : lyrics})


In [12]:
print(rt2)

시간이 지났어도
나 많이 힘들어했어
이제 괜찮을 줄 알았어
미련하게 계속 네 깃결과 구구 소리를 떠올리는걸
이런 날 넌 내 창가에 다시 내려앉아 줄 수 있니
비둘기야 사랑해

이렇게 난 문득
너의 회색 날개를 떠올리며 잠들기도 해
비둘기야 사랑해
이렇게 난 문득
광장 모퉁이 우리 시간에 잠겨 헤어나오질 못하네

사는 게 원래 이렇게
텅 비어 있었니
네가 주던 따스한 체온이 나를 덮쳐서
내 일상 전체를 포근히 잠기게 해
어딜 가봐도 네가 먼저 떠오르잖아
차라리 눈을 감고
네가 부르던 구구 소리를 듣는 게 더 좋아

그때 햇살이 조금 더 따뜻했더라면
이런 말도 안 되는 상상을 해
네가 다시 와줄 거라고
그렇다 믿어서가 아냐
그저 나는 나를 잃어서
어디로 가야 할지 모르고
멈춰서서
빵조각들만 남겨둔 벤치에 앉아 후회하기 바빠
네가 알았으면 날 얼마나
어이없게 봤을까

이제는 널 놓아줘야 하는데
이제는 잊어야 한단 걸 알면서도
그러기엔 함께한 날들이 너무 많아
그 말조차 핑계야
그냥 네가 보고싶은 거잖아

비둘기야 사랑해
이렇게 난 문득
네가 옥상 난간에 앉아 있던 모습을 떠올리며 잠들기도 해
비둘기야 사랑해
이렇게 난 문득
추억의 빵부스러기들에 잠겨 헤어나오질 못하네

괜찮다가도 나는 어쩔 수 없이
네가 보고싶어
지금은 뭘 하고 있을지 궁금해서
너에게 부스럭거리는 소리로 속삭이면
너도 어쩐지 돌아볼 것만 같은데
그건 내 욕심이겠지
오늘 밤도 잠이 오질 않아, 깃털을 다듬는 너를 상상하며 뒤척이지

비둘기야 사랑해
이렇게 난 문득
네가 나에게 다가와 머리를 비비던 모습을 떠올리며 잠들기도 해
비둘기야 사랑해
이렇게 난 문득
추억의 광장에 멈춰 서서 널 그리워하길 멈추질 못하네


In [13]:
from pydantic import BaseModel, ValidationError, Field, field_validator
from typing import Optional

In [14]:
class User(BaseModel):
    name: str
    age: int | None = None
    

In [15]:
User(name='홍길동', age=20)

User(name='홍길동', age=20)

In [16]:
class Product(BaseModel):
    id: int = Field(..., gt=0, description="제품 ID (양수)")
    name: str = Field(..., min_length=1, max_length=100, description='제품명')
    price: float = Field(..., gt=0, description="가격 (양수)")
    # 선택 필드
    description: Optional[str] = Field(None, max_length=500, description="제품 설명")
    stock: int = Field(default=0, ge=0, description="재고 수량 (0 이상)")
   
    # 검증기
    @field_validator('name')
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError('제품명은 공백일 수 없습니다')
        return v.strip()
   
    @field_validator('price')
    @classmethod
    def price_must_be_reasonable(cls, v):
        if v > 1000_0000:
            raise ValueError('가격이 너무 비쌉니다 (최대 1000 만원)')
        return v


In [17]:
from langchain_core.output_parsers import PydanticOutputParser

In [18]:
class transfer(BaseModel):
    text: str = Field(description="사용자가 입력한 값을 영어로 변경한 결과를 출력하는 컬럼")

In [19]:
parser = PydanticOutputParser(pydantic_object=transfer)

In [20]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"text": {"description": "사용자가 입력한 값을 영어로 변경한 결과를 출력하는 컬럼", "title": "Text", "type": "string"}}, "required": ["text"]}\n```'

In [21]:
llm = ChatOpenAI(
    model='gpt-5-mini',
    temperature=0.2,
    reasoning_effort='medium'
)

In [22]:
prompt = PromptTemplate(
    input_variables=['input'],
    partial_variables={
        'format_instructions': parser.get_format_instructions(),
    },
    template=(
        '당신은 번역하는 AI입니다. 아래 사용자 입력 정보를 영어로 번역하세요.\n'
        '{input}\n\n'
        '{format_instructions}'
    )
)

In [23]:
chain2 = prompt | llm | parser

In [24]:
chain2.invoke({'input': "점심은 언제 먹어야 제일 맛있을까?"})

transfer(text='When should I eat lunch to make it taste the best?')

In [25]:
input_text = input("검색하고 싶은 논문은? ")
search_keyword = chain2.invoke({"input": f"{input_text}"}).text

In [26]:
import arxiv
client  = arxiv.Client()


search = arxiv.Search(
    query=search_keyword,
    max_results= 10,
    sort_by=arxiv.SortCriterion.SubmittedDate
)


In [27]:
for result in client.results(search):
    print(f"제목: {result.title}")
    print(f"저자: {[author.name for author in result.authors]}")
    print(f"발행일: {result.published}")
    print(f"URL: {result.entry_id}")
    print(f"요약: {result.summary[:100]}...\n")


HTTPError: Page request resulted in HTTP 301: None (http://export.arxiv.org/api/query?search_query=knowledge+graph&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=10)

In [ ]:
"삼성전자는 대한민국 안의 기업이다." 